<table width=100%>
<tr>
<td width=60%>
<h1><b>MET/MATT Computer Vision with Deep Learning</b></h1>
<h2><b>Lab7 - Motion and Tracking</b></h2>
<br>
<h4>2020-2021 - Josep Ramon Morros
<br>
<a href="https://imatge.upc.edu/web/"> GPI @ IDEAI</a> Research group
<a href="https://telecos.upc.edu/ca">ETSETB – UPC.TelecosBCN</a></h4>
</td>
<td style="text-align: right;" width=40%>
<img src="https://drive.google.com/uc?export=view&id=1tAyPa600X2kvCmWEIPbkdDGFgZPaMziC" width=300>
</td>
</tr>
</table>



## **1. Optical Flow computation**

In this part you will study how to compute optical flow. We will use the C. Liu's [2] implementation of the Brox robust optical flow method [1]. The method uses a multiresolution approach and a  non-linearized version of the brightness-constancy equation, among other details. C. Liu added additional improvements to the method [2].

First, we will install the pyflow github repository, which is a python wrapper to the C++ implementation from C. Liu [3]

In [ ]:
%cd /content
!pip install apng
# To create animated png's
# Clone pyflow repository
!git clone https://github.com/pathak22/pyflow.git
# Build the package
%cd pyflow/
!python setup.py build_ext -i

In [ ]:
import cv2
import pyflow
import numpy as np
import time
from IPython.display import Image # To display images in Colab
from apng import APNG # To create animated png's

In [ ]:
im1 = cv2.imread('examples/car1.jpg')
im2 = cv2.imread('examples/car2.jpg')
im1 = im1.astype(float) / 255.
im2 = im2.astype(float) / 255.

Let's visualize the test images:

In [ ]:
%cd /content/pyflow
APNG.from_files(['examples/car1.jpg', 'examples/car2.jpg'], delay=800).save("result.png")
Image('result.png')

The number of resolution levels is determined using two parameters: ***ratio*** and ***minWidth***. *ratio* determines the reduction in size factor when downsampling the images at each resolution level. *minWidth* determines the width (in pixels) of the lowest level of the pyramid.

Now, we define the OF parameters:

In [ ]:
# Flow Options:
alpha = 0.012
ratio = 0.75
minWidth = 20
nOuterFPIterations = 7
nInnerFPIterations = 1
nSORIterations = 30
colType = 0  # 0 or default:RGB, 1:GRAY (but pass gray image with shape (h,w,1))

Let's perform the optical flow estimation with the selected parameters:

In [ ]:
s = time.time()
u, v, im2W = pyflow.coarse2fine_flow(
    im1, im2, alpha, ratio, minWidth, nOuterFPIterations, nInnerFPIterations,
    nSORIterations, colType)
e = time.time()
print('Time Taken: {:.2f} seconds for image of size ({},{},{})'.format(e - s, im1.shape[0], im1.shape[1], im1.shape[2]))
cv2.imwrite('examples/car2W.jpg', (im2W*255).astype(int))

In the output variable *im2W* we have the compensated image.

<font color='red'><b>P1: Write the code to compute the DFD to evaluate the quality of the optical flow. Use the RMSE measure (Root Mean Square Error) for the DFD.</b></font>

In [ ]:
def dfd_rmse(ima1, ima2):
  return np.sqrt(np.mean((ima2 - ima1)**2))

Compute the DFD value using this set of parameters.

In [ ]:
dfd_val1 = dfd_rmse(im1, im2W)
print ('The DFD (RMSE) is {:.3f}'.format(dfd_val1))

Another way to judge the quality of a flow field is to check that:

*   The brightness constancy assumption is satisfied, namely when you flip back and forth frame 1 and motion compensated frame 2, you shouldn't see any movement.
*   The discontinuity of the flow field agrees with the object boundaries.

For this, we will create an animated png with both frame1 and warped frame 2:

In [ ]:
APNG.from_files(["examples/car2W.jpg", "examples/car1.jpg"], delay=800).save("result.png")
Image('result.png')

<font color='red'><b>Q1: Explain which is the function of the parameter

---

*alpha*</b></font> (Hint: look at the slides of the course: ICV_U4_motion_tracking_v3, slide 53.

<font color='white'>alpha is called the smootheness coefficient. It defines how much spatial smoothness we impose to the flow field. In other terms, it introduces a global constraint of smoothness to solve the aperture problem.

Smoothness Constraint Assumption essentially says is that if we consider a particular neighborhood within the image, we will notice that all pixels in that neighborhood move in the same direction. Therefore, pixels of a neighborhood which are next to each other have very similar optical flow vectors as well. (https://datahacker.rs/013-optical-flow-using-horn-and-schunck-method/#The-Smoothness-Constraint-Assumption)


The cost function minimizes the value for brightness constraint and the value for smoothness constraint. So, the alpha value is basically a weighting factor that decides how much importance needs to be given to either of the two constraints (brightness constraint, smoothness constraint)

 </font>

<font color='red'><b>Q2: Now, change parameter alpha (higher and lower value) and explain how the results differ depending on this parameter.</b></font> (NOTE: use the dfd_rmse() function for quantitative results and the animated png for qualitative results

In [ ]:
alpha = 0.001

s = time.time()
u, v, im2W = pyflow.coarse2fine_flow(
    im1, im2, alpha, ratio, minWidth, nOuterFPIterations, nInnerFPIterations,
    nSORIterations, colType)
e = time.time()
print('Time Taken: {:.2f}s for image of size ({}, {}, {})'.format(e - s, im1.shape[0], im1.shape[1], im1.shape[2]))
cv2.imwrite('examples/car2W.jpg', (im2W*255).astype(int))


In [ ]:
dfd_val1 = dfd_rmse(im1, im2W)
print ('The DFD (RMSE) is {}'.format(dfd_val1))

APNG.from_files(["examples/car2W.jpg", "examples/car1.jpg"], delay=2000).save("result.png")
Image('result.png')

In [ ]:
alpha = 100

s = time.time()
u, v, im2W = pyflow.coarse2fine_flow(
    im1, im2, alpha, ratio, minWidth, nOuterFPIterations, nInnerFPIterations,
    nSORIterations, colType)
e = time.time()
print('Time Taken: {:.2f}s for image of size ({}, {}, {})'.format(e - s, im1.shape[0], im1.shape[1], im1.shape[2]))
cv2.imwrite('examples/car2W.jpg', (im2W*255).astype(int))


In [ ]:
dfd_val1 = dfd_rmse(im1, im2W)
print ('The DFD (RMSE) is {}'.format(dfd_val1))

APNG.from_files(["examples/car2W.jpg", "examples/car1.jpg"], delay=50).save("result.png")
Image('result.png')

<font color='white'>Although the error is practically the same (around 0.12) and time (around 7 seconds). However, the results in the image are different.

- When `alpha = 0.001` we are imposing more weight to the Brightness Constraint. This means that each pixel amost gets its own vector which results in the produced image has some noise in some zones but other static zones are kept static.

- When `alpha = 100` we are imposing more weight to the Smoothness Contraint. This results in a very strong smoothness in the image flow. This is because we are contraining that neighboring vectors have almost the same direction. However, this results in static objects also appear to move.
 </font>


Now, change the number of resolution levels to 5 by modifying either the ratio or minWidth parameters (or both).

In [ ]:
minWidth = 15
ratio    = 0.5
alpha    = 0.012

s = time.time()
u, v, im2W = pyflow.coarse2fine_flow(
    im1, im2, alpha, ratio, minWidth, nOuterFPIterations, nInnerFPIterations,
    nSORIterations, colType)
e = time.time()
print('Time Taken: {:.2f}s for image of size ({}, {}, {})'.format(e - s, im1.shape[0], im1.shape[1], im1.shape[2]))
cv2.imwrite('examples/car2W.jpg', (im2W*255).astype(int))

In [ ]:
dfd_val1 = dfd_rmse(im1, im2W)
print ('The DFD (RMSE) is {}'.format(dfd_val1))

APNG.from_files(["examples/car2W.jpg", "examples/car1.jpg"], delay=50).save("result.png")
Image('result.png')

<font color='red'><b>Q3: Explain the effects on the computation time and on the quality of the optical flow.</b></font>

<font color='white'>As we can see with the previous exeriment, the time is reduced in half and with respect to the `alpha = 0.001` experiment the image is less noisy, with some noise around the object that move. Additionally, the error is 0.029 which is 4 times lower than in the previous experiment. However, since alpha is still low, we are not seeing movement.</font>

## **2. Multi-Object tracking with DeepSORT**

In this part, you will learn how to perform object tracking with the [DeepSORT](https://arxiv.org/pdf/1703.07402.pdf) [5] algorithm.  The method is described by the key components of detection, propagating object states into future frames using a Kalman filter, associating current detections with existing objects, and managing the lifespan of tracked objects. The DeepSORT algorithm is an extension of [SORT](https://arxiv.org/pdf/1602.00763.pdf) [4], a very simple algorithm but with an excellent performance. DeepSORT improves the association step by adding an additional distance metric based on the appearance of the object. The appearance is modelled using a feature vector extracted from a CNN trained for classification. The association integrates motion and appearance information through combination of two appropriate metrics.

TASK:  Read the two papers and answer the following questions (related to the DeepSORT paper):

<font color='red'><b>Q4: Which metric is used to account for the motion information in the association problem?</b></font>

<font color='white'>the metric used is the Mahalanobis distance of the predicted state distribution obtained from Lamna filtering framework. If the distance exceeds a threshold, it is discarded

$$
d_{i,j}^{\mathrm{motion}}
= (d_j - H\,\hat y_i)^\top\,S_i^{-1}\,(d_j - H\,\hat y_i)
$$

The gating indicator:
$$
b^{(1)}_{ij} = \mathbf{1}\bigl[d^{(1)}(i,j)<t^{(1)}\bigr]
$$

</font>

<font color='red'><b>Q5: Which metric is used for the appearance information?</b></font>

<font color='white'>The metric used is feature descriptor obtained from a pre-trained CNN
The cost is the cosine distance between the feature vector and the track's feature
$$
d_{i,j}^{\mathrm{app}}
= \min_{k}\bigl[\,1 - r_j^\top\,r_i^{(k)}\bigr]
$$

where  
- $r_j\in\mathbb{R}^{128}$ is the appearance feature of detection \(j\),  
- $r_i^{(k)}$ is the gallery of past features for track \(i\).

The gating indicator:

$$
b^{(2)}_{ij} = \mathbf{1}\bigl[d^{(2)}(i,j)<t^{(2)}\bigr]
$$


</font>

<font color='red'><b>Q6: How are both metrics combined to provide the final association result?</b></font>

<font color='white'>A weigthed sum

$$
   c_{i,j}
   = \lambda\,d_{i,j}^{\mathrm{motion}}
     \;+\;(1-\lambda)\,d_{i,j}^{\mathrm{app}}
$$
An association is admissible if it is within the gating region of both metrics:
$$
  b_{i,j} = \prod_{m=1}^{2} b_{i,j}^{(m)}
$$
</font>

Now, we will use DeepSORT for person tracking in a video sequence.


The original DeepSORT github repository code is built only for validating the algorithm with the MARS test dataset. We will be using a different [repository](https://github.com/abhyantrika/nanonets_object_tracking/), which includes a custom class deepsort.py that acts as a bridge and also takes in any custom configurations like a different feature extractor and other parameters.

In [ ]:
%cd /content/
!git clone https://github.com/rmorros/deepsort-lab.git
%cd deepsort-lab

In [ ]:
# Change deepsort-lab/deepsort.py to fix errors
# from deep_sort.deep_sort import nn_matching
# from deep_sort.deep_sort.tracker import Tracker
# from deep_sort.application_util import preprocessing as prep
# from deep_sort.deep_sort.detection import Detection

# import numpy as np

# import torch
# import torchvision
# from scipy.stats import multivariate_normal
# from siamese_net import SiameseNetwork


# def get_gaussian_mask():
# 	#128 is image size
# 	x, y = np.mgrid[0:1.0:128j, 0:1.0:128j]
# 	xy = np.column_stack([x.flat, y.flat])
# 	mu = np.array([0.5,0.5])
# 	sigma = np.array([0.22,0.22])
# 	covariance = np.diag(sigma**2)
# 	z = multivariate_normal.pdf(xy, mean=mu, cov=covariance)
# 	z = z.reshape(x.shape)

# 	z = z / z.max()
# 	z  = z.astype(np.float32)

# 	mask = torch.from_numpy(z)

# 	return mask

# class deepsort_rbc():
# 	def __init__(self,wt_path=None):
# 		# register it as “safe” for torch.load
# 		#loading this encoder is slow, should be done only once.
# 		#self.encoder = generate_detections.create_box_encoder("deep_sort/resources/networks/mars-small128.ckpt-68577")
# 		torch.serialization.add_safe_globals([SiameseNetwork])
# 		if wt_path is not None:
# 			self.encoder = torch.load(wt_path, weights_only=False)
# 		else:
# 			self.encoder = torch.load('ckpts/model640.pt', weights_only=False)

# 		self.encoder = self.encoder.cuda()
# 		self.encoder = self.encoder.eval()
# 		print("Deep sort model loaded")

# 		self.metric = nn_matching.NearestNeighborDistanceMetric("cosine",.5 , 100)
# 		self.tracker= Tracker(self.metric)

# 		self.gaussian_mask = get_gaussian_mask().cuda()


# 		self.transforms = torchvision.transforms.Compose([ \
# 			torchvision.transforms.ToPILImage(),\
# 			torchvision.transforms.Resize((128,128)),\
# 			torchvision.transforms.ToTensor()])



# 	def reset_tracker(self):
# 		self.tracker= Tracker(self.metric)
# 		#Deep sort needs the format `top_left_x, top_left_y, width,height

# 	def format_yolo_output(self, out_boxes):
# 		for b in range(len(out_boxes)):
# 			out_boxes[b][0] = out_boxes[b][0] - out_boxes[b][2]/2
# 			out_boxes[b][1] = out_boxes[b][1] - out_boxes[b][3]/2
# 		return out_boxes

# 	def pre_process(self,frame,detections):

# 		transforms = torchvision.transforms.Compose([ \
# 			torchvision.transforms.ToPILImage(),\
# 			torchvision.transforms.Resize((128,128)),\
# 			torchvision.transforms.ToTensor()])

# 		crops = []
# 		for d in detections:

# 			for i in range(len(d)):
# 				if d[i] <0:
# 					d[i] = 0

# 			img_h,img_w,img_ch = frame.shape

# 			xmin,ymin,w,h = d

# 			if xmin > img_w:
# 				xmin = img_w

# 			if ymin > img_h:
# 				ymin = img_h

# 			xmax = xmin + w
# 			ymax = ymin + h

# 			ymin = abs(int(ymin))
# 			ymax = abs(int(ymax))
# 			xmin = abs(int(xmin))
# 			xmax = abs(int(xmax))

# 			try:
# 				crop = frame[ymin:ymax,xmin:xmax,:]
# 				crop = transforms(crop)
# 				crops.append(crop)
# 			except:
# 				continue

# 		crops = torch.stack(crops)

# 		return crops

# 	def extract_features_only(self,frame,coords):

# 		for i in range(len(coords)):
# 			if coords[i] <0:
# 				coords[i] = 0


# 		img_h,img_w,img_ch = frame.shape

# 		xmin,ymin,w,h = coords

# 		if xmin > img_w:
# 			xmin = img_w

# 		if ymin > img_h:
# 			ymin = img_h

# 		xmax = xmin + w
# 		ymax = ymin + h

# 		ymin = abs(int(ymin))
# 		ymax = abs(int(ymax))
# 		xmin = abs(int(xmin))
# 		xmax = abs(int(xmax))

# 		crop = frame[ymin:ymax,xmin:xmax,:]
# 		#crop = crop.astype(np.uint8)

# 		#print(crop.shape,[xmin,ymin,xmax,ymax],frame.shape)

# 		crop = self.transforms(crop)
# 		crop = crop.cuda()

# 		gaussian_mask = self.gaussian_mask

# 		input_ = crop * gaussian_mask
# 		input_ = torch.unsqueeze(input_,0)

# 		features = self.encoder.forward_once(input_)
# 		features = features.detach().cpu().numpy()

# 		corrected_crop = [xmin,ymin,xmax,ymax]

# 		return features,corrected_crop


# 	def run_deep_sort(self, frame, out_scores, out_boxes):

# 		if len(out_boxes)==0:
# 			self.tracker.predict()
# 			print('No detections')
# 			trackers = self.tracker.tracks
# 			return trackers

# 		detections = np.array(out_boxes)
# 		#features = self.encoder(frame, detections.copy())

# 		processed_crops = self.pre_process(frame,detections).cuda()
# 		processed_crops = self.gaussian_mask * processed_crops

# 		features = self.encoder.forward_once(processed_crops)
# 		features = features.detach().cpu().numpy()

# 		if len(features.shape)==1:
# 			features = np.expand_dims(features,0)


# 		dets = [Detection(bbox, score, feature) \
# 					for bbox,score, feature in\
# 				zip(detections,out_scores, features)]

# 		outboxes = np.array([d.tlwh for d in dets])

# 		outscores = np.array([d.confidence for d in dets])
# 		indices = prep.non_max_suppression(outboxes, 0.8,outscores)

# 		dets = [dets[i] for i in indices]

# 		self.tracker.predict()
# 		self.tracker.update(dets)

# 		return self.tracker,dets

However, this code does not include an object detector (for simplicity, a video with pre-existing detection outputs is provided). To be able to use the code for any video, your task will be to modify the code to include an object detector.

You can use any detector available (classical or Deep Learning) which includes the class **person**. For instance, [YOLOv8](https://docs.ultralytics.com/tasks/detect/#predict), [Mobilenet SSD](https://https://www.pyimagesearch.com/2018/05/14/a-gentle-guide-to-deep-learning-object-detection/), [YOLO](https://www.pyimagesearch.com/2018/11/12/yolo-object-detection-with-opencv/) or any other you want.

In [ ]:
import numpy as np
import cv2
import sys
from deepsort import *

from IPython.display import HTML
from base64 import b64encode
import os.path
import warnings

warnings.filterwarnings("ignore")


# Read Video sequence
if not os.path.isfile('videos/race.mp4'):
  print ('Can not find videos/race.mp4')
  exit

cap = cv2.VideoCapture('videos/race.mp4')

# Initialize writer
fourcc = cv2.VideoWriter_fourcc(*'MP4V')
out = cv2.VideoWriter('race_out.mp4',fourcc, 10.0, (1280,720))

#Initialize deep sort.
deepsort = deepsort_rbc()


<font color='red'><b>P2: Modify the following code to include an object detector. Draw a rectangle around each tracked object, using a different box color for each track.
Test the result on the provided video</b></font>

In [ ]:
#TODO: Initialize object detector
# !pip install ultralytics
from ultralytics import YOLO

# Load a model
model = YOLO("yolo12n.pt")  # Load an official Detect model

frame_id = 1

while True:
    print('Processing frame {}'.format(frame_id))
    ret,frame = cap.read()
    if ret is False:
        frame_id+=1
        break
    frame = frame.astype(np.uint8)

    # detections, out_scores = get_gt(frame, frame_id, gt_dict)
    # TODO: Get detections,out_scores from an object detector
    results = model(frame, classes=0)[0]
    detections = results.boxes.xyxy.cpu().numpy()   # x1,y1,x2,y2
    out_scores = results.boxes.conf.cpu().numpy()

    if len(detections) == 0:
        print("No dets")
        frame_id+=1
        continue

    tracker,detections_class = deepsort.run_deep_sort(frame,out_scores,detections)

    for track in tracker.tracks:
        if not track.is_confirmed() or track.time_since_update > 1:
            continue

        bbox = track.to_tlbr() #Get the corrected/predicted bounding box
        id_num = str(track.track_id) #Get the ID for the particular track.
        features = track.features #Get the feature vector corresponding to the detection.

        # Overlay bbox from tracker.
        cv2.rectangle(frame, (int(bbox[0]), int(bbox[1])), (int(bbox[2]), int(bbox[3])),(255,255,255), 2)
        cv2.putText(frame, str(id_num),(int(bbox[0]), int(bbox[1])),0, 5e-3 * 200, (0,255,0),2)

    frame_id+=1
    out.write(frame)

# When everything done, release the capture
out.release()

Visualize the video with the overlayed detections to check that it works:

In [ ]:
# Transcode to h264, otherwise video is not visualizing in Colab
!ffmpeg -i race_out.mp4 -an -vcodec libx264 -crf 23 race_out2.mp4 -y

# Extracted from https://stackoverflow.com/questions/57377185/how-play-mp4-video-in-google-colab
mp4 = open('race_out2.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=800 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

<font color='red'><b>Q7: Comment the obtained results: Which is the tracking rate (fps)? Are the labels of the objects maintained along the video sequence? In which situations does the method fail? Can you use the tracking to count the number of people in the race? Which is the frame-rate of the tracking?</b></font>

<font color='white'>

- The results are not very satisfactory as the bounding boxes are very big, even overlapping with other boxes. Additionally, the id tracking is not well mantined.
- Input video is at fps 10
- The labels are not always maintained as for example the 3 runner from the right, alternates ids, for example, 11 and 15. Or ids are assigned to other runners.
- As we can see, we end up with ids such as 89, 79, 64, 35 etc. We can see that the method fails to detect in cases of eclusions. Also on when the angle of the players change, the tracking of ids is lost almost completely
- I would not use this tracking method as bounding boxes are not very precise and some players in some occasions are not detected.
- Detections are not always mantained when there is occlusion. For example on the first seconds of the video, when player with bounding box `id = 6` ocludes `id = 10` the bounding box for `id = 10` is not detected.
</font>


**REFERENCES**

[1] *T. Brox, A. Bruhn, N. Papenberg, and J.Weickert. High accuracy optical flow estimation based on a theory for warping. In European Conference on Computer Vision (ECCV), pages 25–36, 2004*

[2] *C. Liu. Beyond Pixels: Exploring New Representations and Applications for Motion Analysis. Doctoral Thesis. Massachusetts Institute of Technology. May 2009.*

[3] *C. Liu, "[Optical Flow Matlab/C++ Code](http://people.csail.mit.edu/celiu/OpticalFlow/)"*

[4] *Bewley, Alex et al. “Simple Online and Realtime Tracking.” 2016 IEEE International Conference on Image Processing (ICIP), 2016*

[5] *Nicolai Wojke and Alex Bewley and Dietrich Paulus, Simple Online and Realtime Tracking with a Deep Association Metric, arXiv 1703.07402, 2017*


**USEFUL RESOURCES**

*   [pyimagesearch Blog](https://www.pyimagesearch.com/blog)
*   [Nanonets.com Blog](https://nanonets.com/blog/object-tracking-deepsort/)



